# Exercise 5: Simple Market

**Presenter** <br>
Priyesh Gosai <br>
Energy Systems Modeller <br>
priyesh@innovateimpact.com <br>

**Course Details**<br>
Modelling Integrated Power Markets <br>
Johannesburg, South Africa <br>
20-24 April 2026 <br>


In [ ]:
lesson_number = 4
print(f'lesson{lesson_number}')

## Prepare Google Colab Environment

In [ ]:
#@title Connect to Google Drive {display-mode:"form"}
CONNECT_TO_DRIVE = False #@param {type:"boolean"}

import os

if CONNECT_TO_DRIVE:
    from google.colab import drive
    # Mount Google Drive
    drive.mount('/content/drive')

    # Define the desired working directory path
    working_dir = f'/content/drive/MyDrive/ich-modelling-2026'
    lesson_folder = f'lesson-{lesson_number}'
    lesson_dir = os.path.join(working_dir, lesson_folder)

    # Create the working directory if it doesn't exist
    if not os.path.exists(working_dir):
        os.makedirs(working_dir)
        print(f"Directory '{working_dir}' created.")
    else:
        print(f"Directory '{working_dir}' already exists.")

    # Create the lesson directory if it doesn't exist
    if not os.path.exists(lesson_dir):
        os.makedirs(lesson_dir)
        print(f"Directory '{lesson_dir}' created.")
    else:
        print(f"Directory '{lesson_dir}' already exists.")

    # Change the current working directory to the lesson directory
    os.chdir(lesson_dir)

    print(f"Current working directory: {os.getcwd()}")
else:
    print("Not connecting to Google Drive.")

In [ ]:
#@title Install Packages {display-mode:"form"}
INSTALL_PACKAGES = False #@param {type:"boolean"}

import os

# Check if packages have already been installed in this session to prevent re-installation
if INSTALL_PACKAGES and not os.environ.get('PYPSA_PACKAGES_INSTALLED'):
  !pip install git+https://github.com/pypsa/pypsa
  !pip install pypsa[excel] 
  !pip install folium mapclassify cartopy gurobipy
  os.environ['PYPSA_PACKAGES_INSTALLED'] = 'true'
elif not INSTALL_PACKAGES:
  print("Skipping package installation.")
else:
  print("PyPSA packages are already installed for this session.")

In [ ]:
#@title Download the file for this notebook {display-mode:"form"}
DOWNLOAD_FILE = False #@param {type:"boolean"}

import urllib.request
import os

if DOWNLOAD_FILE:
    url = "https://raw.githubusercontent.com/PriyeshGosai/ich_course_2026/46e109dd68c635d83046eec03d2d2a4afc669366/n-01-single-node-v2.xlsx"
    filename = url.split('/')[-1]  # Extract filename from URL
    
    urllib.request.urlretrieve(url, filename)
    print(f"File downloaded successfully: {os.path.abspath(filename)}")

else:
    print("Skipping file download.")

## Functions

In [ ]:
def setup_gurobi_license_file():
    from google.colab import userdata
    import os
    
    wls_access_id = userdata.get('WLSACCESSID')
    wls_secret = userdata.get('WLSSECRET')
    license_id = userdata.get('LICENSEID')
    
    license_content = f"""WLSACCESSID={wls_access_id}
                        WLSSECRET={wls_secret}
                        LICENSEID={license_id}
                        """
    
    os.makedirs('/opt/gurobi', exist_ok=True)
    with open('/opt/gurobi/gurobi.lic', 'w') as f:
        f.write(license_content)
    
    print("✓ License file created")
    print(f"  License ID: {license_id}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def plot_supply_demand_curve(supplier_volume, supplier_prices, consumer_volume, consumer_prices, bus_name, snapshot_idx, n):
    """
    Plot supply and demand curves for a given bus and snapshot.
    
    Parameters:
    - supplier_volume: DataFrame of supplier volumes
    - supplier_prices: DataFrame of supplier prices
    - consumer_volume: DataFrame of consumer volumes
    - consumer_prices: DataFrame of consumer prices
    - bus_name: Name of the bus to plot
    - snapshot_idx: Index of the snapshot to plot
    - n: PyPSA network object
    """
    
    # Supply curve
    supply_at_bus = supplier_volume.iloc[snapshot_idx][supplier_volume.iloc[snapshot_idx] > 0]
    supply_prices_at_bus = supplier_prices.loc[n.snapshots[snapshot_idx], supply_at_bus.index]
    
    supply_curve = pd.DataFrame({
        'dispatch': supply_at_bus.values,
        'price': supply_prices_at_bus.values
    }, index=supply_at_bus.index)
    
    supply_curve = supply_curve[supply_curve['dispatch'] > 0].sort_values('price')
    
    # Demand curve
    demand_at_bus = consumer_volume.iloc[snapshot_idx][consumer_volume.iloc[snapshot_idx] > 0]
    demand_prices_at_bus = consumer_prices.loc[n.snapshots[snapshot_idx], demand_at_bus.index]
    
    demand_curve = pd.DataFrame({
        'dispatch': demand_at_bus.values,
        'price': demand_prices_at_bus.values
    }, index=demand_at_bus.index)
    
    demand_curve = demand_curve[demand_curve['dispatch'] > 0].sort_values('price', ascending=False)
    
    # Calculate max for xlim
    max_dispatch = max(supply_curve['dispatch'].sum(), demand_curve['dispatch'].sum())
    
    # Plot
    plt.figure(figsize=(10, 6))
    
    plt.step(
        np.cumsum([0] + supply_curve['dispatch'].tolist()),
        supply_curve['price'].iloc[:1].tolist() + supply_curve['price'].tolist(),
        where='pre',
        linewidth=2,
        color='steelblue',
        label='Supply'
    )
    
    if len(demand_curve) > 0:
        plt.step(
            np.cumsum([0] + demand_curve['dispatch'].tolist()),
            demand_curve['price'].iloc[:1].tolist() + demand_curve['price'].tolist(),
            where='pre',
            linewidth=2,
            color='coral',
            label='Demand'
        )
    
    plt.xlim(0, max_dispatch)
    plt.xlabel('Cumulative Dispatch (MW)')
    plt.ylabel('Price (ZAR/MWh)')
    plt.title(f'Supply-Demand Curve - {bus_name} at {n.snapshots[snapshot_idx]}(Snapshot {snapshot_idx})')
    plt.legend()
    plt.tight_layout()
    plt.show()




In [ ]:
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import pandas as pd
from datetime import datetime

# ============================================================================
# MATPLOTLIB VERSION
# ============================================================================

def plot_dispatch_matplotlib(n, start_date=None, end_date=None, aggregate_data=True):
    """
    Plot dispatch with Matplotlib.
    
    Parameters:
    -----------
    n : pypsa.Network
        The PyPSA network object containing the data to plot.
    start_date : str, optional
        Start date in format 'YYYY-MM-DD' (always 00:00)
    end_date : str, optional
        End date in format 'YYYY-MM-DD' (always 23:00)
    aggregate_data : bool, default True
        If True, aggregate all generators/storage units.
        If False, show individual plant contributions.
    """
    
    # 1. Get generator names (excluding elastic demand)
    non_elastic_generators = n.generators.static[n.generators.static.sign >= 0].index
    
    # 2. Split dispatch and storage
    gen_dispatch = n.generators.dynamic.p[non_elastic_generators]  # Exclude elastic demand
    storage_discharge = n.storage_units.dynamic.p[n.storage_units.dynamic.p >= 0]
    storage_charge = n.storage_units.dynamic.p[n.storage_units.dynamic.p < 0].abs()

    # 3. Prepare data - convert charging to negative
    charging = -storage_charge.fillna(0)  # Negative for below x-axis
    generators = gen_dispatch.fillna(0)
    discharging = storage_discharge.fillna(0)

    # 4. Get load data
    fixed_load = n.loads.dynamic.p.sum(axis=1)
    elastic_generators = n.generators.static[n.generators.static.sign < 0].index
    elastic_load = n.generators.dynamic.p[elastic_generators].sum(axis=1)

    # Filter data by date range
    if start_date and end_date:
        start_dt = pd.Timestamp(start_date + ' 00:00')
        end_dt = pd.Timestamp(end_date + ' 23:00')
        
        mask = (charging.index >= start_dt) & (charging.index <= end_dt)
        charging_filtered = charging[mask]
        generators_filtered = generators[mask]
        discharging_filtered = discharging[mask]
        fixed_load_filtered = fixed_load[mask]
        elastic_load_filtered = elastic_load[mask]
    else:
        charging_filtered = charging
        generators_filtered = generators
        discharging_filtered = discharging
        fixed_load_filtered = fixed_load
        elastic_load_filtered = elastic_load
    
    # Create figure
    fig, ax = plt.subplots(figsize=(14, 6))
    
    # Stacked area plot
    if aggregate_data:
        # Aggregated view
        ax.stackplot(
            charging_filtered.index,
            charging_filtered.sum(axis=1),  # Negative
            generators_filtered.sum(axis=1),
            discharging_filtered.sum(axis=1),
            labels=['Charging Storage Units', 'Generator Dispatch', 'Discharging Storage Units'],
            alpha=0.7,
            colors=['#1f77b4', '#ff7f0e', '#2ca02c']
        )
    else:
        # Individual plant view
        # Charging storage units (negative)
        charging_sum = charging_filtered.sum(axis=1)
        ax.fill_between(charging_filtered.index, 0, charging_sum, alpha=0.7, color='#1f77b4', label='Charging Storage Units')
        
        # Generator dispatch (individual)
        ax.stackplot(
            generators_filtered.index,
            *[generators_filtered[col] for col in generators_filtered.columns],
            alpha=0.7,
            labels=generators_filtered.columns
        )
        
        # Discharging storage units (individual, stacked above generators)
        gen_cumsum = generators_filtered.sum(axis=1)
        for col in discharging_filtered.columns:
            ax.fill_between(discharging_filtered.index, gen_cumsum, gen_cumsum + discharging_filtered[col], 
                           alpha=0.7, label=col)
            gen_cumsum = gen_cumsum + discharging_filtered[col]
    
    # Overlay loads
    load_total = fixed_load_filtered + elastic_load_filtered
    ax.plot(fixed_load_filtered.index, fixed_load_filtered, color='black', linewidth=2.5, label='Total Fixed Load', zorder=5)
    ax.plot(load_total.index, load_total, color='red', linewidth=2.5, linestyle='--', label='Total Load (Fixed + Elastic)', zorder=5)
    
    # Formatting
    ax.axhline(y=0, color='black', linewidth=0.8, zorder=4)
    ax.set_xlabel('Time', fontsize=12)
    ax.set_ylabel('Power (MW)', fontsize=12)
    ax.set_title('Dispatch with Storage Charging/Discharging and Load', fontsize=14)
    ax.set_xlim(charging_filtered.index.min(), charging_filtered.index.max())
    ax.legend(loc='upper left', fontsize=9, bbox_to_anchor=(1.01, 1))
    ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


# ============================================================================
# PLOTLY VERSION
# ============================================================================

def plot_dispatch_plotly(n, aggregate_data=True):
    """
    Plot dispatch with Plotly (interactive).
    
    Parameters:
    -----------
    n : pypsa.Network
        The PyPSA network object containing the data to plot.
    aggregate_data : bool, default True
        If True, aggregate all generators/storage units.
        If False, show individual plant contributions.
    """
    
    # 1. Get generator names (excluding elastic demand)
    non_elastic_generators = n.generators.static[n.generators.static.sign >= 0].index
    
    # 2. Split dispatch and storage
    gen_dispatch = n.generators.dynamic.p[non_elastic_generators]  # Exclude elastic demand
    storage_discharge = n.storage_units.dynamic.p[n.storage_units.dynamic.p >= 0]
    storage_charge = n.storage_units.dynamic.p[n.storage_units.dynamic.p < 0].abs()

    # 3. Prepare data - convert charging to negative
    charging = -storage_charge.fillna(0)  # Negative for below x-axis
    generators = gen_dispatch.fillna(0)
    discharging = storage_discharge.fillna(0)

    # 4. Get load data
    fixed_load = n.loads.dynamic.p.sum(axis=1)
    elastic_generators = n.generators.static[n.generators.static.sign < 0].index
    elastic_load = n.generators.dynamic.p[elastic_generators].sum(axis=1)

    fig = go.Figure()
    
    if aggregate_data:
        # Aggregated view
        fig.add_trace(go.Scatter(
            x=charging.index,
            y=charging.sum(axis=1),
            mode='lines',
            name='Charging Storage Units',
            stackgroup='one',
            fillcolor='rgba(31, 119, 180, 0.7)',
            line=dict(color='rgba(31, 119, 180, 0.7)')
        ))
        
        fig.add_trace(go.Scatter(
            x=generators.index,
            y=generators.sum(axis=1),
            mode='lines',
            name='Generator Dispatch',
            stackgroup='one',
            fillcolor='rgba(255, 127, 14, 0.7)',
            line=dict(color='rgba(255, 127, 14, 0.7)')
        ))
        
        fig.add_trace(go.Scatter(
            x=discharging.index,
            y=discharging.sum(axis=1),
            mode='lines',
            name='Discharging Storage Units',
            stackgroup='one',
            fillcolor='rgba(44, 160, 44, 0.7)',
            line=dict(color='rgba(44, 160, 44, 0.7)')
        ))
    else:
        # Individual plant view
        # Charging storage units (negative, non-stacking)
        for col in charging.columns:
            fig.add_trace(go.Scatter(
                x=charging.index,
                y=charging[col],
                mode='lines',
                name=f'{col} (Charging)',
                stackgroup='charging',
                fillcolor='rgba(31, 119, 180, 0.7)',
                line=dict(color='rgba(31, 119, 180, 0.7)')
            ))
        
        # Generator dispatch (individual, stacked positive)
        for col in generators.columns:
            fig.add_trace(go.Scatter(
                x=generators.index,
                y=generators[col],
                mode='lines',
                name=col,
                stackgroup='generation',
                fillcolor=None,
                line=dict(width=1)
            ))
        
        # Discharging storage units (individual, stacked positive)
        for col in discharging.columns:
            fig.add_trace(go.Scatter(
                x=discharging.index,
                y=discharging[col],
                mode='lines',
                name=f'{col} (Discharging)',
                stackgroup='generation',
                fillcolor=None,
                line=dict(width=1)
            ))
    
    # Overlay loads
    fig.add_trace(go.Scatter(
        x=fixed_load.index,
        y=fixed_load,
        mode='lines',
        name='Total Fixed Load',
        line=dict(color='black', width=2.5),
        yaxis='y'
    ))
    
    load_total = fixed_load + elastic_load
    fig.add_trace(go.Scatter(
        x=load_total.index,
        y=load_total,
        mode='lines',
        name='Total Load (Fixed + Elastic)',
        line=dict(color='red', width=2.5, dash='dash'),
        yaxis='y'
    ))
    
    # Layout
    fig.update_layout(
        title='Dispatch with Storage Charging/Discharging and Load',
        xaxis_title='Time',
        yaxis_title='Power (MW)',
        hovermode='x unified',
        height=600,
        template='plotly_white'
    )
    
    fig.show()

#### Case Description

Simulate least cost dispatch given fixed capacity availability with unit commitment and rolling horizon optimization.

##### PyPSA Features Used

| Feature | Method |
|---------|--------|
| Inspect dispatch data | Programmatically |
| Activate unit commitment for dispatchable generators | Programmatically |
| Run rolling horizon optimisation | Programmatically |
| Calculate marginal prices | Programmatically |
| Plot supply and demand | Programmatically |

##### Non-Standard PyPSA Features

This module only covers standard PyPSA features.

#### Lesson Tasks

1. Inspect market model results
2. Compare results with and without unit commitment
3. Compare how rolling horizon optimisation differs from full period optimisation

## Model

### User inputs

In [ ]:
file_name = 'n_02_single_node_v2.xlsx'
solver_name = 'gurobi'  # 'gurobi' or 'highs'


### Import packages, read input data and optimize

In [ ]:
# Import packages
import gurobipy as gp
import pypsa
import pandas as pd
import linopy
import plotly.graph_objects as go

import linopy.solvers
linopy.solvers.gurobipy = gp


# Supress unnecessary warnings
pd.set_option('future.no_silent_downcasting', True)

# The default plotting tool for pandas is matplotlib which only provies static plots. 
# Changing to plotly allows for interactive plotting. 
pd.set_option('plotting.backend', 'plotly')

# When pypsa version 1 was launched, a new set of commands were created. 
# This option ensures that we use the new commands. 
pypsa.options.api.new_components_api = True

Run five cases with the following options:




| Case # | Unit <br> Commitment | Rolling Horizon <br> Optimization | Horizon | Overlap | Month|
|--------|-----------------|------------------------------|---------|---------|---------|
| 1      | False           | False                        | -       | -       | False|
| 2      | False           | False                        | -       | -       | 1-12*|
| 3      | True            | False                        | -       | -       | 1-12*|
| 4      | True            | True                         | 168     | 144     | 1-12*|
| 5      | True            | True                         | 168     | 0       | 1-12*|
| 6      | True            | True                         | 72      | 48      | 1-12*|

*Select any month of the year. 

In [ ]:
month = False # False : Insert month number (1-12) or False for full year simulation


unit_commitment = True # True or False for unit commitment of dispatchable generators (coal and gas in this case)

rolling_horizon_optimization = False # True or False for rolling horizon optimization (only relevant if unit_commitment is True)

rolling_horizon = 168
overlap = 144 # 168-24


In [ ]:
# Model import and optimization
print(f'Load Network from {file_name}')
n = pypsa.Network(file_name)

# Import metadata (optional)
n.meta = pd.read_excel(file_name, sheet_name='meta', index_col=0, header=None)[1].to_dict()



In [ ]:
# Reduce snapshots to month if specified (based on user inputs)
if month is not False: n.set_snapshots(n.snapshots[n.snapshots.month == month])

# set committable to True for dispatchable generators
dispatchable_carriers = ['coal', 'gas']
n.generators.static.loc[n.generators.static.carrier.isin(dispatchable_carriers), 'committable'] = unit_commitment

if rolling_horizon_optimization:
    n.optimize_with_rolling_horizon(horizon=rolling_horizon, overlap=overlap, include_objective_constant=True)
else:
    n.optimize(include_objective_constant = True, solver_name = solver_name)

### Post Process

In [ ]:
# Step 1: Separate volumes into suppliers and consumers
supplier_volume = pd.concat([
    n.generators.dynamic.p[n.generators.static[n.generators.static.type == 'Supply'].index],
    n.storage_units.dynamic.p[n.storage_units.dynamic.p >= 0].fillna(0)
], axis=1)

consumer_volume = pd.concat([
    n.loads.dynamic.p[n.loads.static[n.loads.static.type == 'Demand'].index],
    n.generators.dynamic.p[n.generators.static[n.generators.static.type == 'Demand'].index],
    n.storage_units.dynamic.p[n.storage_units.dynamic.p < 0].fillna(0)
], axis=1)

consumer_volume = consumer_volume.abs()

# Step 2: Get marginal costs for all generators and storage units
supplier_prices = pd.DataFrame(index=n.snapshots)
for gen in n.generators.static[n.generators.static.type == 'Supply'].index:
    if gen in n.generators.dynamic.marginal_cost.columns:
        supplier_prices[gen] = n.generators.dynamic.marginal_cost[gen]
    else:
        supplier_prices[gen] = n.generators.static.loc[gen, 'marginal_cost']

for storage in n.storage_units.static.index:
    if storage in n.storage_units.dynamic.marginal_cost.columns:
        supplier_prices[storage] = n.storage_units.dynamic.marginal_cost[storage]
    else:
        supplier_prices[storage] = n.storage_units.static.loc[storage, 'marginal_cost']

# Step 3: Find marginal cost (highest price among dispatched suppliers)
marginal_prices = supplier_prices[supplier_volume > 0].max(axis=1)

# Step 4: All consumers pay the marginal cost
consumer_prices = pd.DataFrame(index=n.snapshots)
for col in consumer_volume.columns:
    consumer_prices[col] = marginal_prices

# Step 5: Prepare offer and bid tables 
supplier_offers = pd.DataFrame(index=n.snapshots)

for gen in n.generators.static[n.generators.static['type'] == 'Supply'].index:
    if gen in n.generators.dynamic.p_max_pu.columns:
        supplier_offers[gen] = n.generators.dynamic.p_max_pu[gen] * n.generators.static.loc[gen, 'p_nom']
    else:
        supplier_offers[gen] = 1 * n.generators.static.loc[gen, 'p_nom']

for gen in n.generators.static[n.generators.static['type'] == 'Supply'].index:
    if gen in n.generators.dynamic.p_max_pu.columns:
        supplier_offers[gen] = n.generators.dynamic.p_max_pu[gen] * n.generators.static.loc[gen, 'p_nom']
    else:
        supplier_offers[gen] = 1 * n.generators.static.loc[gen, 'p_nom']


# Step 5: Calculate Curtailment using statistics library
curtailment = n.statistics.curtailment(carrier = ['solar','wind'],groupby_time = False).fillna(0)
curtailment = curtailment.xs('Generator', level=0).T # The curtailment results use the timestamp as columns, therefore we transpose the table (swop columns with rows).

### Dispatch results

In [ ]:
n.generators.dynamic.p.plot()

In [ ]:
n.storage_units.dynamic.p.plot()

In [ ]:
pd.concat([n.generators.dynamic.p, n.storage_units.dynamic.p], axis=1).plot()

In [ ]:
n.generators.dynamic.status.plot()

In [ ]:
supplier_volume.plot.area(stacked=True)

In [ ]:

consumer_volume.plot.area(stacked=True)

In [ ]:
curtailment.plot(title = 'Renewables Curtailment')

#### Market Results

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=list(range(len(marginal_prices))),
    y=marginal_prices.values,
    mode='lines',
    hovertemplate='<b>Snapshot %{x}</b><br>Price: R%{y:.2f}/MWh<extra></extra>'
))

fig.update_layout(title='Marginal Prices with Timestamp')
fig.show()

View specific snapshots by adjusting the `snapshot_index`

We generate two plots: 
1. Using the offers (which in this case offers are the the marginal price and nominal power)
2. Using the dispatched volumes and market prices.


In [ ]:
# Call the function
snapshot_index = 30

plot_supply_demand_curve(supplier_offers, supplier_prices, consumer_volume, consumer_prices, 'Western Cape', snapshot_index, n)

plot_supply_demand_curve(supplier_volume, supplier_prices, consumer_volume, consumer_prices, 'Western Cape', snapshot_index, n)

In [ ]:
supplier_data = pd.DataFrame({
    'Offers (MWh)': supplier_offers.iloc[snapshot_index, :],
    'Volume (MWh)': supplier_volume.iloc[snapshot_index, :],
    'Prices (R/MWh)': supplier_prices.iloc[snapshot_index, :],
})
    
consumer_data = pd.DataFrame({
    'Volume (MWh)': consumer_volume.iloc[snapshot_index, :],
    'Prices (R/MWh)': consumer_prices.iloc[snapshot_index, :],
})

# Create multi-index
supplier_data.index = pd.MultiIndex.from_product([['Supplier'], supplier_data.index])
consumer_data.index = pd.MultiIndex.from_product([['Consumer'], consumer_data.index])

# Combine
results = pd.concat([supplier_data, consumer_data], axis=0)
results = results.fillna(0)

print(f'Results for index - {snapshot_index}: {n.snapshots[snapshot_index]}')
display(results)

print()
print(f'Renewables curtailment (MWh) for index - {snapshot_index}: {n.snapshots[snapshot_index]}')
print(curtailment.iloc[snapshot_index,:].to_string())